In [0]:
%run ./transform

In [0]:
# 1. Base Class (Blueprint)
class DataSink:
    def __init__(self, df, path, method, params):
        self.df = df
        self.path = path
        self.method = method
        self.params = params

    def load_data_frame(self):
        raise ValueError("Not Implemented")

# 2. Standard Load (Delta Added)
class LoadToDBFS(DataSink):
    def load_data_frame(self):
        # .format("delta") yahan add kiya gaya hai
        self.df.write.format("delta").mode(self.method).save(self.path)

# 3. Partitioned Load (Delta Added)
class LoadToDBFSWithPartition(DataSink):
    def load_data_frame(self):
        PartitionByColumns = self.params.get('PartitionByColumns')
        # .format("delta") yahan bhi add kiya gaya hai
        self.df.write.format("delta").mode(self.method).partitionBy(*PartitionByColumns).save(self.path)

# 4. 
class LoadToDeltaTable(DataSink):
    def load_data_frame(self):
        # 'saveasTable' ko 'saveAsTable' (Capital A and T) se badal dein
        self.df.write.format("delta").mode(self.method).saveAsTable(self.path)


# 5. Factory Function
def get_sink_source(sink_type, df, path, method, params=None): 
    if sink_type == 'dbfs':
        return LoadToDBFS(df, path, method, params) 
    elif sink_type == 'dbfs_partition':
        return LoadToDBFSWithPartition(df, path, method, params)
    elif sink_type == 'delta':
        return LoadToDeltaTable(df, path, method, params)
    else:
        raise ValueError(f"Invalid sink type: {sink_type}")